# 07 — Contextual Retrieval: does a written context line help a chunk?

**Concept:** before a chunk is embedded, prepend a short line that says where it comes from.
[Anthropic's Contextual Retrieval post](https://www.anthropic.com/engineering/contextual-retrieval)
has an LLM write that line from the whole document. The free control is to prepend the title.

## The prediction, written before running (2026-10-29)

> **The LLM context line beats plain chunks, but not the title.** Arm 3's paired nDCG@10 CI
> against arm 1 lies above 0. Its CI against arm 2 includes 0.
>
> NFCorpus documents are short abstracts (about 1.7k chars). What a chunk loses when it is cut
> out is mostly the title, and the title is free.

The prediction fails if arm 3 against arm 1 includes 0, or if arm 3 against arm 2 excludes 0 in
either direction.

## Setup, fixed before running

Every arm uses notebook 05's chunk-pool document vector and changes only the text of each chunk.
Same encoder (`all-MiniLM-L6-v2`), same chunks (400-char windows of the body, 80 overlap, 18,835
chunks over 3,633 docs), same blend (0.25 × title vector + 0.75 × mean of chunk vectors).

| arm | text embedded for each chunk |
|---|---|
| 1. plain | the chunk (notebook 05's `chunk_pool`; recomputed here and checked against the committed scores) |
| 2. title | the doc title, a newline, the chunk (docs with an empty title get no prefix) |
| 3. LLM context | a Claude Haiku context line, a newline, the chunk |

Arm 3's lines come from headless Claude Code (`claude -p --model haiku`), one call per document.
The call sees the title, the abstract and the numbered chunks. It returns one line per chunk,
following the post's prompt ("a short succinct context to situate this chunk within the overall
document for the purposes of improving search retrieval"). Every line is committed under
`results/context/`, so the notebook reruns without an LLM. A document whose reply cannot be parsed
after two retries keeps its plain chunks, and the count of those is reported.

Contextual BM25 is left out: BM25 lost to dense in notebook 03.

Scored on all 323 queries against the human qrels. `paired_bootstrap`, seed 42, 10,000 resamples,
corpus order. Reported alongside but not bet on: arm 2 against arm 1, and every arm against the
title+text dense baseline and against title-only.
